In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import regex as re

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving pizza.txt to pizza (1).txt


In [ ]:
def file_to_sentence_list(file_path):
    with open(file_path, 'r') as file:
        text = file.read()

    # Splitting the text into sentences using
    # delimiters like '.', '?', and '!'
    sentences = [sentence.strip() for sentence in re.split(
        r'(?<=[.!?])\s+', text) if sentence.strip()]

    return sentences

file_path = 'pizza.txt'
text_data = file_to_sentence_list(file_path)

# Tokenize the text data
tokenizer = Tokenizer()
tokenizer.fit_on_texts(text_data)
total_words = len(tokenizer.word_index) + 1

# Create input sequences
input_sequences = []
for line in text_data:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences and split into predictors and label
max_sequence_len = max([len(seq) for seq in input_sequences])
input_sequences = np.array(pad_sequences(
    input_sequences, maxlen=max_sequence_len, padding='pre'))
X, y = input_sequences[:, :-1], input_sequences[:, -1]

# Convert target data to one-hot encoding
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

In [ ]:
# Define the model
model = Sequential()
model.add(Embedding(total_words, 10,
                    input_length=max_sequence_len-1))
model.add(LSTM(128))
model.add(Dense(total_words, activation='softmax'))
model.compile(loss='categorical_crossentropy',
              optimizer='adam', metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# Train the model
model.fit(X, y, epochs=200, verbose=1)

Epoch 1/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - accuracy: 0.0366 - loss: 6.3430
Epoch 2/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 49ms/step - accuracy: 0.0570 - loss: 5.7700
Epoch 3/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 4s 68ms/step - accuracy: 0.0555 - loss: 5.7134
Epoch 4/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.0631 - loss: 5.6782
Epoch 5/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.0537 - loss: 5.6195
Epoch 6/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.0562 - loss: 5.6224
Epoch 7/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 63ms/step - accuracy: 0.0569 - loss: 5.5175
Epoch 8/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 54ms/step - accuracy: 0.0567 - loss: 5.5199
Epoch 9/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 5s 52ms/step - accuracy: 0.0637 - loss: 5.4253
Epoch 10/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.0726 - loss: 5.3374
Epoch 11/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 4s 71ms/step - accuracy: 0.0776 - loss: 5.2738
Epoch 12/200
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step

In [ ]:
# Generate next word predictions
seed_text = "Pizza have different "
next_words = 5

for _ in range(next_words):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences(
        [token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted_probs = model.predict(token_list)
    predicted_word = tokenizer.index_word[np.argmax(predicted_probs)]
    seed_text += " " + predicted_word

print("Next predicted words:", seed_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 219ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
Next predicted words: Pizza have different  become a symbol of comfort


In [ ]:
new_seed_text = "I love to eat"
new_next_words = 7

current_text = new_seed_text
for _ in range(new_next_words):
    token_list = tokenizer.texts_to_sequences([current_text])[0]
    token_list = pad_sequences(
        [token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted_probs = model.predict(token_list, verbose=0)
    # Get the index of the predicted word
    predicted_word_index = np.argmax(predicted_probs)
    # Handle cases where the predicted_word_index might not be in tokenizer.index_word
    if predicted_word_index in tokenizer.index_word:
        predicted_word = tokenizer.index_word[predicted_word_index]
    else:
        predicted_word = "_UNKNOWN_"
    current_text += " " + predicted_word

print("Generated text:", current_text)

Generated text: I love to eat the united states the deep dish pizza


Feel free to change the `new_seed_text` and `new_next_words` variables in the code cell above to generate different sequences!

# Task
I'll create a Python script to enable interactive text generation. You'll be able to input a seed phrase, and the model will predict the next word. You can accept the prediction, override it with your own word, or type 'quit' to exit the interactive mode. This will demonstrate how the model predicts words based on your input and how you can control the text generation process.

## Implement Interactive Text Generation

### Subtask:
Create a Python script that continuously takes user input as a seed phrase, predicts the next word using the trained model, and displays the generated text. The user should be able to press Enter to accept the predicted word, type their own word to override, or type 'quit' to exit the interactive mode.


**Reasoning**:
I will create a Python script for interactive text generation based on the provided instructions. This script will continuously take user input for seed phrases, predict next words using the trained model, and allow the user to accept predictions, override them, or quit.



In [ ]:
print("\n--- Interactive Text Generation ---")
print("Type 'quit' at any prompt to exit.")

while True:
    seed_text_input = input("\nEnter a starting seed phrase (or 'quit' to exit): ")
    if seed_text_input.lower() == 'quit':
        break

    current_text = seed_text_input
    print(f"Current generated text: {current_text}")

    print("\n--- Generating next words ---")
    while True:
        token_list = tokenizer.texts_to_sequences([current_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted_probs = model.predict(token_list, verbose=0)
        predicted_word_index = np.argmax(predicted_probs)

        predicted_word = tokenizer.index_word.get(predicted_word_index, "_UNKNOWN_")

        print(f"Generated text: {current_text} [{predicted_word}]")
        user_choice = input(f"Press Enter to accept '{predicted_word}', type your own word, or 'quit': ")

        if user_choice.lower() == 'quit':
            print("Exiting current generation sequence.\n")
            break
        elif user_choice == '':
            current_text += " " + predicted_word
        else:
            current_text += " " + user_choice

        # Optional: Add a condition to stop inner loop if text becomes too long or model struggles
        # For this example, it continues until user quits inner loop



--- Interactive Text Generation ---
Type 'quit' at any prompt to exit.

Enter a starting seed phrase (or 'quit' to exit): hello
Current generated text: hello

--- Generating next words ---
Generated text: hello [and]
Press Enter to accept 'and', type your own word, or 'quit': sir
Generated text: hello sir [and]
Press Enter to accept 'and', type your own word, or 'quit': i
Generated text: hello sir i [and]
Press Enter to accept 'and', type your own word, or 'quit': am
Generated text: hello sir i am [and]
Press Enter to accept 'and', type your own word, or 'quit': kuwar 
Generated text: hello sir i am kuwar  [and]
Press Enter to accept 'and', type your own word, or 'quit': 
Generated text: hello sir i am kuwar  and [let's]
Press Enter to accept 'let's', type your own word, or 'quit': 
Generated text: hello sir i am kuwar  and let's [not]
Press Enter to accept 'not', type your own word, or 'quit': quit
Exiting current generation sequence.



KeyboardInterrupt: Interrupted by user